In [88]:
import torch.nn as nn
import torch 
from torch.utils.data import Dataset,DataLoader
from torchvision.io import read_image
import cv2

In [89]:
if torch.cuda.is_available():
    device = torch.device("cuda")

In [90]:
device

device(type='cuda')

In [91]:
def add_gaussian_noise_normal(tensor, mean=0., sigma=0.1):
    # Generate noise with the specified mean, std, and the shape of the input tensor
    noise = torch.normal(mean, sigma, size=tensor.shape, device=tensor.device, dtype=tensor.dtype)
    noisy_tensor = noise #+tensor
    # Optionally, clip the values to a valid range
    # noisy_tensor = torch.clamp(noisy_tensor, 0., 1.)
    return noisy_tensor

In [92]:
def conv(in_chs, out_chs):
	return nn.Sequential(
		nn.Conv2d(in_channels=in_chs, out_channels=out_chs, kernel_size=3, padding=1),
		nn.BatchNorm2d(out_chs),
		nn.ReLU(inplace=True),
		nn.Conv2d(in_channels=out_chs, out_channels=out_chs, kernel_size=3, padding=1),
		nn.BatchNorm2d(out_chs),
		nn.ReLU(inplace=True)
	)

In [93]:
class encoder_block(nn.Module):
	def __init__(self, in_chs, out_chs):
		super(encoder_block, self).__init__()
		self.encode=conv(in_chs, out_chs)
		self.max_pool=nn.MaxPool2d(kernel_size=2, stride=2)
	def forward(self, X):
		X=self.encode(X)
		X_Pooled=self.max_pool(X)
		return X_Pooled,X


In [94]:
def deconv(in_chs, out_chs):
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels=in_chs, out_channels=out_chs, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_chs),
        nn.ReLU(inplace=True),
        nn.ConvTranspose2d(in_channels=out_chs, out_channels=out_chs, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_chs),
        nn.ReLU(inplace=True)
    )

In [95]:
def upsample(in_chs, out_chs):
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels=in_chs, out_channels=out_chs, kernel_size=6, stride=2, padding=2),
        nn.BatchNorm2d(out_chs),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4)
    )

In [96]:
class decoder_block(nn.Module):
    def __init__(self, in_chs, out_chs):
        super(decoder_block, self).__init__()
        self.upsampling=upsample(in_chs, out_chs)
        self.deconv=nn.Sequential(
            deconv(in_chs, out_chs),
            deconv(out_chs, out_chs)
        )
    def forward(self, X, X_skip):
        X=self.upsampling(X)
        X=self.deconv(torch.cat([X,X_skip],dim=1))
        return X

In [97]:
class encoder(nn.Module):
    def __init__(self, in_chs, out_chs):
        super(encoder, self).__init__()
        
        self.encoder_blocks=nn.ModuleList([encoder_block(in_chs,32)])

        cur_ch=32
        while cur_ch < out_chs:
            self.encoder_blocks.append(encoder_block(cur_ch,cur_ch*2))
            cur_ch*=2
    def forward(self, X):
        cur_X=X
        X_Skips=[]
        for layer in self.encoder_blocks:
            X,X_Skip=layer(cur_X)
            #X_Skips.append(X_Skip)
            X_Skips.append(add_gaussian_noise(X_Skip))
            cur_X=X
        return X,X_Skips

In [98]:
class decoder(nn.Module):
    def __init__(self, in_chs, out_chs):
        super(decoder, self).__init__()
        self.decoder_blocks=nn.ModuleList()
        cur_ch=in_chs
        while cur_ch>out_chs:
            self.decoder_blocks.append(decoder_block(cur_ch,cur_ch//2))
            cur_ch=cur_ch//2
    def forward(self, X, X_Skips):
        cur_idx=len(X_Skips)-1
        for layer in self.decoder_blocks:
            if cur_idx<0:
                break
            X=layer(X,add_gaussian_noise_normal(X_Skips[cur_idx]))
            #X=layer(X,X_Skips[cur_idx])
            cur_idx-=1
        return X
        

In [99]:
class U_Net(nn.Module):
    def __init__(self):
        super(U_Net, self).__init__()
        self.encoder_part=encoder(3,256)
        self.bottle_neck=nn.Sequential(
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1)
        )
        self.decoder_part=decoder(512,32)
        self.final=nn.Conv2d(in_channels=32, out_channels=3, kernel_size=1, padding=0)
    def forward(self,X):
        X,X_Skips=self.encoder_part(X)
        X=self.bottle_neck(X)
        X=self.decoder_part(X,X_Skips)
        X=self.final(X)
        return X
        

In [100]:
PATH=r"C:\Users\ADMIN\Documents\Bitches generator\models\best_model.pth"

In [101]:
model=U_Net().to(device)
model.load_state_dict(torch.load(PATH, weights_only=True))

<All keys matched successfully>

In [102]:
img = cv2.imread(r"C:\Users\ADMIN\Documents\Bitches generator\test\test1.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (512, 512))
img = img.astype("float32")

img = torch.from_numpy(img).permute(2, 0, 1)
img = img.unsqueeze(0).to(device)        

In [103]:
import matplotlib.pyplot as plt
import torch

model.eval()

with torch.no_grad():
    X = model(img)  # (1,3,512,512)

# reshape về (H,W,C)
img_vis = img.squeeze(0).permute(1, 2, 0).cpu().numpy()
out_vis = X.squeeze(0).permute(1, 2, 0).cpu().numpy()

# clip về [0,255] để hiển thị đúng
img_vis = img_vis.clip(0, 255).astype("uint8")
out_vis = out_vis.clip(0, 255).astype("uint8")

# plot
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.title("Original")
plt.imshow(img_vis)
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Output")
plt.imshow(out_vis)
plt.axis("off")

plt.show()

NameError: name 'add_gaussian_noise' is not defined